In [ ]:
!pip install transformers datasets

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

In [41]:
class Config:
    '''
    Class for model configuration
    vocab_size = V, 
    max_seq_length = T,
    embed_size = D,
    num_layers = 12,
    num_heads = 12,
    dropout = 0.1
    '''
    def __init__(self, vocab_size = 50257, max_seq_length = 128, embed_size = 768,
                 num_layers = 12, num_heads = 12, dropout = 0.1):
        self.vocab_size = vocab_size
        self.max_seq_length = max_seq_length
        self.embed_size = embed_size
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.dropout =dropout

In [49]:
class MultiHeadSelfAttention(nn.Module):
    '''
    Multihead self attention
    embed_size % num_heads = 0
    '''
    def __init__(self, config):
        super().__init__()
        # embed_size % num_heads = 0
        assert config.embed_size % config.num_heads == 0, 'Dimensions do not match'
        self.num_heads = config.num_heads
        self.head_dim = config.embed_size // config.num_heads
        # Multi head self attention
        self.W_q = nn.Linear(config.embed_size, config.embed_size) #12 W_qs, where each is shape (embed_size, head_dim)
        self.W_k = nn.Linear(config.embed_size, config.embed_size)
        self.W_v = nn.Linear(config.embed_size, config.embed_size)
        self.output = nn.Linear(config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        #lower triangular matrix for causal attention
        self.register_buffer(
            'mask',
            torch.tril(torch.ones(config.max_seq_length, config.max_seq_length)
                      ).view(-1, 1, config.max_seq_length, config.max_seq_length))
    def forward(self, x):
        batch, seq_length, embed_dim = x.size() # MB, T, D => 16, 128, 768
        # Compute multi head Q, K, V
        # x@W_q.T => (-1, 128, 768) @ (768, 768).T => (-1, 128, 768)
        Q = self.W_q(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) 
        # (-1, 128, 768) => (batch, seq_length, num_heads, head_dim) => (batch, num_heads, seq_legth, head_dim) (-1, 12, 128, 64)
        K = self.W_k(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) 
        V = self.W_v(x).view(batch, seq_length, self.num_heads, self.head_dim).transpose(1, 2) 
        attention = (Q@K.transpose(-2, -1)) / (self.head_dim ** 0.5) #(-1, 12, 128, 128)
        # (batch, num_heads, seq_legth, head_dim)@(batch, num_heads, head_dim, seq_legth) => (batch, num_heads, seq_legth, seq_length)
        # causal attention
        attention = attention.masked_fill(self.mask[:, :, :seq_length, :seq_length] == 0, float('-inf'))
        #(batch, num_heads, seq_legth, seq_length)
        attention = F.softmax(attention, dim = -1)
        #(batch, num_heads, seq_legth, seq_length)
        attention = self.dropout(attention)
        scores = attention @ V # (batch, num_heads, seq_legth, seq_length) @ ((batch, num_heads, seq_legth, head_dim)
        #(batch, num_heads, seq_legth, head_dim)
        scores = scores.transpose(1, 2).contiguous().view(batch, seq_length, embed_dim)
        #(batch, seq_legth, num_heads, head_dim) = > (batch, seq_legth, embed_dim)
        scores = self.output(scores)
        return self.dropout(scores)
        

In [50]:
class FFN(nn.Module):
    '''
    Feed forward neural network
    '''
    def __init__(self, config):
        super().__init__()
        self.fc1 = nn.Linear(config.embed_size, 4 * config.embed_size)
        self.gelu = nn.GELU()
        self.fc2 = nn.Linear(4 * config.embed_size, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc2(self.gelu(self.fc1(x)))
        return self.dropout(x)

In [51]:
class Transformer(nn.Module):
    '''
    Build transformer block
    '''
    def __init__(self, config):
        super().__init__()
        self.norm1 = nn.LayerNorm(config.embed_size)
        self.attention = MultiHeadSelfAttention(config)
        self.norm2 = nn.LayerNorm(config.embed_size)
        self.mlp = FFN(config)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

In [52]:
class GPT2(nn.Module):
    '''
    Building GPT2
    '''
    def __init__(self, config):
        super().__init__()
        self.token_embed = nn.Embedding(config.vocab_size, config.embed_size) #(vocab_size, embed_size)
        self.pos_embed = nn.Embedding(config.max_seq_length, config.embed_size)
        self.dropout = nn.Dropout(config.dropout)
        self.transfomers = nn.Sequential(*[Transformer(config) for _ in range(config.num_layers)])
        self.norm1 = nn.LayerNorm(config.embed_size)

    def forward(self, input_tokens):
        batch, seq_length = input_tokens.size()
        pos = torch.arange(0, seq_length, dtype = torch.long, device = input_tokens.device).unsqueeze(0)
        x = self.token_embed(input_tokens) + self.pos_embed(pos) #(batch, seq_length, embed_size)
        x = self.dropout(x)
        x = self.transfomers(x)
        x = self.norm1(x)     ##(batch, seq_length, embed_size)
        return x @ self.token_embed.weight.t()
        

In [53]:
import requests
# Download Don Quijote 
url = "https://www.gutenberg.org/cache/epub/2000/pg2000.txt"
response = requests.get(url)
text = response.text

In [54]:
from transformers import GPT2TokenizerFast

In [55]:
tokeniser = GPT2TokenizerFast.from_pretrained("gpt2")
tokeniser.pad_token = tokeniser.eos_token
tokens = tokeniser.encode(text)
data = torch.tensor(tokens, dtype = torch.long)

Token indices sequence length is longer than the specified maximum sequence length for this model (860018 > 1024). Running this sequence through the model will result in indexing errors


In [56]:
class Quijote(Dataset):
    def __init__(self, data, seq_length):
        self.text = data
        self.seq_length = seq_length

    def __len__(self):
        return len(self.text) - self.seq_length

    def __getitem__(self, idx):
        x = self.text[idx : idx + self.seq_length]
        y = self.text[idx + 1: idx + 1 + self.seq_length]
        return x, y
        
        

In [57]:
SEQ_LENGTH = 128

In [58]:
quijote_dataset = Quijote(data, SEQ_LENGTH)

In [59]:
quijote_loader = DataLoader(quijote_dataset, batch_size = 16, shuffle= True)

In [60]:
config = Config(vocab_size=tokeniser.vocab_size,
                max_seq_length= SEQ_LENGTH,
                embed_size= 768,
                num_layers= 12,
                num_heads= 12,
                dropout=0.1)

In [61]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT2(config).to(device)

In [66]:
optimiser = optim.Adam(model.parameters(), lr=3e-4)

In [62]:
def sample(model, device, tokeniser, prompt, length = 50, 
           temperature = 1.0):
    model.eval()
    tokens = tokeniser.encode(prompt, return_tensors = 'pt').to(device)
    for _ in range(length):
        tokens_ = tokens[:, -SEQ_LENGTH:]
        with torch.no_grad():
            scores = model(tokens_)

        next_token_scores = scores[:, -1, :] / temperature
        next_token = torch.multinomial(F.softmax(next_token_scores, dim = -1), num_samples = 1)
        tokens = torch.cat([tokens, next_token], dim = 1)

    return tokeniser.decode(tokens[0])


In [64]:
print(sample(model, device, tokeniser, 'en un lugar de la mancha del cual', temperature=2.0))

en un lugar de la mancha del cualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualual


In [65]:
def train(model, loader, optimiser, epochs = 20):
    model.train()
    for epoch in range(epochs):
        total_loss = 0.0
        for i, (x, y) in enumerate(loader):
            x = x.to(device)
            y = y.to(device)
            optimiser.zero_grad()
            scores = model(x)
            loss = F.cross_entropy(scores.view(-1, scores.size(-1)), y.view(-1))
            loss.backward()
            optimiser.step()

            total_loss += loss.item()

            if i % 200 == 0:
                print(f'epoch: {epoch + 1}, step:{i}, loss: {loss.item():.4f}')
                print(sample(model, device, tokeniser, 'en un lugar de la mancha del cual', temperature=0.7))
                print()
        epoch_loss = total_loss/len(loader)
        print(f'epoch: {epoch + 1}, loss: {epoch_loss:.4f}')
    

In [67]:
train(model, quijote_loader, optimiser, 5)

epoch: 1, step:0, loss: 451.0701
en un lugar de la mancha del cualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualualual

epoch: 1, step:200, loss: 10.9381
en un lugar de la mancha del cualualualónón suo de suenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaenciaoososadoadoadoadoa.
 p p p p p po.
aleg al nu d

epoch: 1, step:400, loss: 6.4031
en un lugar de la mancha del cual su de la de la Donás de la christaba, y, yoderCEía, aula de la


 hacer, que todos a vuesto de, y.

la,

epoch: 1, step:600, loss: 5.3090
en un lugar de la mancha del cuales
azón de yo yo las
a de que decir de la buenaena. Aaceombo a Sancho Panza y violenceon que porque es de la cual con la con vuestra mer

epoch: 1, step:800, loss: 4.9491
en un lugar de la mancha del cualesen las puedo que
de unañas que me falta a la herbrada, enros, para sobre que, y no lo que, le debía la caballero, y encant

KeyboardInterrupt: 

In [71]:
torch.save({
    'config': config.__dict__,
    'state_dict': model.state_dict()
}, "gpt2_model_and_config.pth")


In [68]:
torch.save(model.state_dict(), "gpt2_260725_weights.pth")

In [70]:
tokeniser.save_pretrained("my_tokenizer/")

('my_tokenizer/tokenizer_config.json',
 'my_tokenizer/special_tokens_map.json',
 'my_tokenizer/vocab.json',
 'my_tokenizer/merges.txt',
 'my_tokenizer/added_tokens.json',
 'my_tokenizer/tokenizer.json')

In [ ]:
model = GPT2Model(config)
model.load_state_dict(torch.load("gpt2_260725_weights.pth"))
